#  **Movie Recommendation System using Advanced RAG**
> **Architecture**: Two-Stage Retrieval (Bi-Encoder + Cross-Encoder Reranking) with 4-bit Quantized Mistral-7B.

## 1. Environment Setup & Libraries Installation
> **Objective**: Install and import the core dependencies for data processing, vector search, reranking, and LLM generation.

In [1]:
!pip install -U sentence-transformers
!pip install -U chromadb
!pip install -q langchain-text-splitters nltk
!pip install -U bitsandbytes accelerate

In [2]:
# 1. Data & Core Utilities
import pandas as pd
import torch
from huggingface_hub import hf_hub_download

# 2. Text Processing & Embeddings
import nltk

nltk.download("punkt_tab")
nltk.download("punkt")

from langchain_text_splitters import NLTKTextSplitter
from sentence_transformers import SentenceTransformer

# 3. Vector Database
import chromadb

# 4. LLM & Generation
import transformers
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    BitsAndBytesConfig,
    pipeline
)

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


## 2. Data Loading & Preprocessing
> **Objective**: Acquire and prepare the dataset for our Retrieval-Augmented Generation (RAG) system.

In this step, we download the **TMDB-5000** movie dataset directly from HuggingFace.
To optimize our vector search and keep the context clean, we filter the dataset to retain only the essential features:
- `original_title`: Used to identify the movie.
- `overview`: The text block that will be embedded and searched.

In [3]:
# Download from the public tmdb-5000 repo
file_path = hf_hub_download(
    repo_id="alejandrowallace/tmdb-5000",
    filename="tmdb_5000_movies.csv",
    repo_type="dataset",
)

# Load into pandas and filter
df = pd.read_csv(file_path)
df = df[["original_title", "overview"]].dropna().reset_index(drop=True)

# Save locally as movies.csv for downstream tutorial steps
df.to_csv("movies.csv", index=False)

print(f"Loaded successfully! Shape: {df.shape}")
print(df.head())

Loaded successfully! Shape: (4800, 2)
                             original_title  \
0                                    Avatar   
1  Pirates of the Caribbean: At World's End   
2                                   Spectre   
3                     The Dark Knight Rises   
4                               John Carter   

                                            overview  
0  In the 22nd century, a paraplegic Marine is di...  
1  Captain Barbossa, long believed to be dead, ha...  
2  A cryptic message from Bond’s past sends him o...  
3  Following the death of District Attorney Harve...  
4  John Carter is a war-weary, former military ca...  


## 3. Text Chunking
> **Objective**: Process long text into manageable semantic blocks to fit within the embedding model's context window.

By splitting long movie overviews into chunks using `NLTKTextSplitter`, we preserve sentence boundaries and semantic meaning without overwhelming the embedding model.

In [4]:
df = df[['original_title', 'overview']]

# Chunk the Overview Column using NLTKTextSplitter
text_splitter = NLTKTextSplitter(chunk_size=1500)

def split_overview(overview):
    if pd.isna(overview):
        return []
    return text_splitter.split_text(str(overview))

df['chunks'] = df['overview'].apply(split_overview)

# Flatten the dataframe for easier processing
chunked_df = df.explode('chunks').reset_index(drop=True)

## 4. Vectorization (Generating Embeddings)
> **Objective**: Map text chunks into a high-dimensional dense vector space.

We use the `all-MiniLM-L6-v2` Bi-Encoder model from `sentence-transformers` to compute 384-dimensional embeddings for each movie chunk, hardware-accelerated via GPU.

In [5]:
# Load SentenceTransformer Bi-Encoder on GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
embed_model = SentenceTransformer("all-MiniLM-L6-v2", device=device)

# Ensure chunks are valid non-empty strings
chunked_df = chunked_df[chunked_df["chunks"].apply(lambda x: isinstance(x, str) and x.strip() != "")].reset_index(drop=True)

# Encode all chunks in batches for fast GPU processing
embeddings = embed_model.encode(
    chunked_df["chunks"].tolist(),
    batch_size=64,
    show_progress_bar=True
)

chunked_df["embeddings"] = embeddings.tolist()

print(f"Embeddings generated successfully! Total records: {len(chunked_df)}")
display(chunked_df[["original_title", "chunks", "embeddings"]].head(3))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/75 [00:00<?, ?it/s]

Embeddings generated successfully! Total records: 4799


,original_title,chunks,embeddings
0,Avatar,"In the 22nd century, a paraplegic Marine is di...","[-0.006365567445755005, 0.06156245991587639, 0..."
1,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[-0.040365736931562424, -0.02718244306743145, ..."
2,Spectre,A cryptic message from Bond’s past sends him o...,"[-0.08593849092721939, 0.02028488926589489, -0..."


## 5. Vector Storage (ChromaDB)
> **Objective**: Index and store the generated embeddings for fast semantic retrieval.

We initialize a local **ChromaDB** client and create a `movies` collection using **Cosine Similarity** (`hnsw:space: cosine`). Each record stores:
- **ID**: Unique string identifier.
- **Document**: The movie overview chunk.
- **Embedding**: The 384-d dense vector.
- **Metadata**: Movie title and chunk text for context construction.

In [6]:
# Initialize ChromaDB client and collection
client = chromadb.Client()

# Reset collection if it already exists
try:
    client.delete_collection("movies")
except Exception:
    pass

collection = client.create_collection(
    name="movies",
    metadata={"hnsw:space": "cosine"}
)

# Insert records in batches (ChromaDB recommended practice)
batch_size = 1000
for start_idx in range(0, len(chunked_df), batch_size):
    end_idx = min(start_idx + batch_size, len(chunked_df))
    batch = chunked_df.iloc[start_idx:end_idx]

    collection.add(
        ids=[str(i) for i in batch.index],
        documents=batch["chunks"].tolist(),
        embeddings=batch["embeddings"].tolist(),
        metadatas=[
            {"original_title": title, "chunk": chunk}
            for title, chunk in zip(batch["original_title"], batch["chunks"])
        ]
    )

print(f"Data successfully stored in ChromaDB! Total indexed items: {collection.count()}")

Data successfully stored in ChromaDB! Total indexed items: 4799


## 6. Load Cross-Encoder Reranker (MonoBERT)
> **Objective**: Improve retrieval precision using a second-stage Cross-Encoder.

While Bi-Encoders (`all-MiniLM-L6-v2`) are fast for initial candidate retrieval, a Cross-Encoder (`MonoBERT`) processes the query and document simultaneously through self-attention, assigning a highly accurate relevance score.

In [7]:
reranker_model_name = "castorini/monobert-large-msmarco-finetune-only"

reranker_tokenizer = AutoTokenizer.from_pretrained(reranker_model_name)
reranker_model = AutoModelForSequenceClassification.from_pretrained(
    reranker_model_name,
    device_map="auto"
)
reranker_model.eval()

print("MonoBERT Reranker loaded successfully!")

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

MonoBERT Reranker loaded successfully!


## 7. Load Quantized LLM (Mistral-7B-Instruct)
> **Objective**: Load a powerful generative model within standard GPU VRAM limits.

Loading a 7B parameter model in `fp16` requires ~14GB+ of VRAM, which can cause Out-Of-Memory (OOM) errors on free-tier GPUs (like Colab T4). Using `bitsandbytes` **4-bit NormalFloat (NF4) Quantization** reduces the memory footprint to ~4.5GB while preserving generation quality.

In [8]:
llm_model_name = "mistralai/Mistral-7B-Instruct-v0.1"

tokenizer = AutoTokenizer.from_pretrained(llm_model_name)
tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    llm_model_name,
    device_map="auto",
    quantization_config=bnb_config
)

text_generation_pipeline = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False
)

print("Mistral-7B-Instruct (4-bit) loaded successfully!")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Mistral-7B-Instruct (4-bit) loaded successfully!


## 8. Stage 1: Initial Candidate Retrieval (Bi-Encoder)
> **Objective**: Perform a fast dense vector search in ChromaDB to fetch the top `K` candidate movie chunks.

In [9]:
# 1. Stage 1: Fast Bi-Encoder Retrieval from ChromaDB
def retrieve_documents(query, collection, top_k=10):
    query_embedding = embed_model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )

    if not results["documents"] or not results["documents"][0]:
        print("No results found for the query.")
        return []

    documents = []
    for metadata, distance in zip(results["metadatas"][0], results["distances"][0]):
        documents.append({
            "title": metadata["original_title"],
            "chunk": metadata["chunk"],
            "vector_distance": round(distance, 4)
        })

    return documents

## 9. Stage 2: Cross-Encoder Reranking (MonoBERT)
> **Objective**: Re-score the retrieved candidates using MonoBERT to evaluate the deep semantic relevance between the query and each movie plot.

In [10]:
def rerank_documents(query, documents, top_k=5):
    if not documents:
        return []

    # Include both title and plot chunk so MonoBERT has full context
    pairs = [(query, f"Movie Title: {doc['title']}. Plot: {doc['chunk']}") for doc in documents]

    # Tokenize query-document pairs
    inputs = reranker_tokenizer(
        [pair[0] for pair in pairs],
        [pair[1] for pair in pairs],
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors="pt"
    )

    # Move tensors to the model's device
    model_device = next(reranker_model.parameters()).device
    inputs = {k: v.to(model_device) for k, v in inputs.items()}

    # Compute relevance scores
    with torch.no_grad():
        outputs = reranker_model(**inputs)
        # Class 1 represents relevance log-probability in MonoBERT
        log_probs = torch.log_softmax(outputs.logits, dim=-1)[:, 1]

    for idx, (doc, log_prob) in enumerate(zip(documents, log_probs.cpu().tolist()), start=1):
        doc["initial_rank"] = idx
        doc["rerank_score"] = round(log_prob, 4)

    # Sort candidates by MonoBERT relevance score
    reranked = sorted(documents, key=lambda x: x["rerank_score"], reverse=True)
    return reranked[:top_k]

## 10. Stage 3: Context-Grounded Answer Generation (Mistral-7B)
> **Objective**: Inject the top reranked movie plots into the Mistral-7B prompt to Generate a context-grounded recommendation using the reranked movie plots.

In [11]:
def generate_answer(query, documents, text_generation_pipeline):
    # Format retrieved context
    context = "\n\n".join([
        f"Movie Title: {doc['title']}\nPlot Overview: {doc['chunk']}"
        for doc in documents
    ])

    prompt = f"""[INST]
You are an expert movie recommendation assistant.
Use ONLY the provided movie context below to answer the user's request.
For each recommended movie, explain why it matches the user's request based on its plot overview.

User Request:
{query}

Context:
{context}
[/INST]"""

    # Configure generation settings cleanly to avoid transformers warnings
    gen_config = model.generation_config
    gen_config.max_length = None
    gen_config.max_new_tokens = 512
    gen_config.do_sample = False
    gen_config.pad_token_id = tokenizer.eos_token_id

    # Generate final response
    output = text_generation_pipeline(
        prompt,
        generation_config=gen_config,
        clean_up_tokenization_spaces=False
    )

    return output[0]["generated_text"].strip()

## 11. Run the End-to-End RAG Pipeline
> **Objective**: Test the complete system on a plot-specific user query: Retrieve Top-10 candidates, Rerank to Top-5, and Generate the final response.

### Step 1: Retrieve Top-10 Candidates from ChromaDB

In [12]:
query = "What are some good science fiction movies about space exploration, astronauts, or alien planets?"

# Retrieve top-10 candidates using Bi-Encoder vector search
retrieved_documents = retrieve_documents(query, collection, top_k=10)

retrieved_df = pd.DataFrame(retrieved_documents)
retrieved_df.index = range(1, len(retrieved_df) + 1)
retrieved_df.index.name = "Initial Rank"

pd.set_option("display.max_colwidth", 180)
print(f"User Query: {query}\n")
display(retrieved_df[["title", "vector_distance", "chunk"]])

User Query: What are some good science fiction movies about space exploration, astronauts, or alien planets?



,title,vector_distance,chunk
Initial Rank,,,
1,Interstellar,0.5110,Interstellar chronicles the adventures of a group of explorers who make use of a newly discovered wormhole to surpass the limitations on human space travel and conquer the vast...
2,In the Shadow of the Moon,0.5488,"Archival material from the original NASA film footage – much of it seen for the first time – plus interviews with the surviving astronauts, including Jim Lovell, Dave Scott, Jo..."
3,Galaxy Quest,0.5787,The stars of a 1970s sci-fi show - now scraping a living through re-runs and sci-fi conventions - are beamed aboard an alien spacecraft.\n\nBelieving the cast's heroic on-scree...
4,Gattaca,0.5838,Science fiction drama about a future society in the era of indefinite eugenics where humans are set on a life course depending on their DNA.\n\nThe young Vincent Freeman is bor...
5,You Only Live Twice,0.5887,A mysterious space craft kidnaps a Russian and American space capsule and brings the world on the verge of another World War.\n\nJames Bond investigates the case in Japan and m...
6,Aliens in the Attic,0.5961,"It's summer vacation, but the Pearson family kids are stuck at a boring lake house with their nerdy parents.\n\nThat is until feisty, little, green aliens crash-land on the roo..."
7,Red Planet,0.6024,"Astronauts search for solutions to save a dying Earth by searching on Mars, only to have the mission go terribly awry."
8,Mars Attacks!,0.6052,"'We come in peace' is not what those green men from Mars mean when they invade our planet, armed with irresistible weapons and a cruel sense of humor.\n\nThis star studded cast..."
9,Oceans,0.6199,"An ecological drama/documentary, filmed throughout the globe.\n\nPart thriller, part meditation on the vanishing wonders of the sub-aquatic world."


### Step 2: Rerank Candidates with MonoBERT (Top-5)
Notice how the Cross-Encoder re-orders the initial candidates based on deep semantic relevance to the query.

In [13]:
# Rerank candidates and keep top-5
reranked_documents = rerank_documents(query, retrieved_documents, top_k=5)

reranked_df = pd.DataFrame(reranked_documents)
reranked_df.index = range(1, len(reranked_df) + 1)
reranked_df.index.name = "Final Rank"

display(reranked_df[["title", "initial_rank", "rerank_score", "vector_distance", "chunk"]])

,title,initial_rank,rerank_score,vector_distance,chunk
Final Rank,,,,,
1,Red Planet,7,-4.6090,0.6024,"Astronauts search for solutions to save a dying Earth by searching on Mars, only to have the mission go terribly awry."
2,Interstellar,1,-5.2561,0.5110,Interstellar chronicles the adventures of a group of explorers who make use of a newly discovered wormhole to surpass the limitations on human space travel and conquer the vast...
3,Galaxy Quest,3,-5.3218,0.5787,The stars of a 1970s sci-fi show - now scraping a living through re-runs and sci-fi conventions - are beamed aboard an alien spacecraft.\n\nBelieving the cast's heroic on-scree...
4,In the Shadow of the Moon,2,-5.4426,0.5488,"Archival material from the original NASA film footage – much of it seen for the first time – plus interviews with the surviving astronauts, including Jim Lovell, Dave Scott, Jo..."
5,Mars Attacks!,8,-6.4497,0.6052,"'We come in peace' is not what those green men from Mars mean when they invade our planet, armed with irresistible weapons and a cruel sense of humor.\n\nThis star studded cast..."


### Step 3: Generate Final Recommendation with Mistral-7B

In [14]:
answer = generate_answer(query, reranked_documents, text_generation_pipeline)

print("=" * 80)
print(f"Question: {query}")
print("=" * 80)
print(f"\nAnswer:\n{answer}")

Question: What are some good science fiction movies about space exploration, astronauts, or alien planets?

Answer:
Based on the user's request for science fiction movies about space exploration, astronauts, or alien planets, here are some recommended movies:

1. Red Planet: This movie follows a group of astronauts who travel to Mars in search of solutions to save a dying Earth. The plot matches the user's request for a movie about space exploration and astronauts.

2. Interstellar: This movie follows a group of explorers who use a newly discovered wormhole to travel vast distances in space. The plot matches the user's request for a movie about space exploration and alien planets.

3. Galaxy Quest: This movie follows a group of actors who are beamed aboard an alien spacecraft and must help the aliens in their quest to overcome an oppressive regime in their solar system. The plot matches the user's request for a movie about astronauts and alien planets.

4. In the Shadow of the Moon: Th